# Capítulo 3 · Proyecto: preparar los datos del Titanic

**EDA, limpieza, transformación y escalado**

**Integrantes:** Oscar Gomez Vasquez, 
Gustavo Emir Jimenez Cruz, 
Eros Arturo Garcia Acosta, 
Fernando Martinez Gomez.

| Fase | Tiempo |
|---|---|
| 1 · Primer contacto | 20 min |
| 2 · Análisis exploratorio | 40 min |
| 3 · Limpieza | 65 min |
| 4 · Transformación y escalado | 40 min |
| 5 · Bitácora y conclusiones | 15 min |

**Reglas:** funciona con *Restart & Run All* · imports al inicio · rutas relativas (`data/titanic.csv`) · `random_state=42` · cada gráfica lleva 1–2 frases de interpretación · limpia las salidas antes de hacer commit.

## Pregunta del proyecto

> Queremos predecir **si un pasajero sobrevivió** (`survived`: 0 = no, 1 = sí) usando **solo los datos que se conocían al abordar** (clase, sexo, edad, familia a bordo, tarifa y puerto). El modelo será útil si supera la **línea base del 62 %**: el acierto que se obtiene diciendo que nadie sobrevivió.

- **Tipo de problema:** clasificación binaria, porque el objetivo es una categoría (sí/no), no una cantidad.
- **Qué dejamos fuera:** columnas que solo se conocen después del resultado o que lo revelan, como `alive` (*data leakage*).
- **Para qué sirve cada fase:** entender los datos (1–2), corregir sus problemas (3) y dejarlos listos para el modelo (4), anotando cada decisión en la bitácora (5).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

SEED = 42
np.random.seed(SEED)

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "titanic.csv")
df.head()

| Columna | Significado | Columna | Significado |
|---|---|---|---|
| `survived` | Objetivo (0 = no, 1 = sí) | `class` | Clase, en texto |
| `pclass` | Clase: 1, 2 o 3 | `who` | Hombre, mujer o niño |
| `sex` | Sexo | `adult_male` | ¿Hombre adulto? |
| `age` | Edad en años | `deck` | Cubierta del camarote |
| `sibsp` | Hermanos / cónyuges a bordo | `embark_town` | Puerto, en texto |
| `parch` | Padres / hijos a bordo | `alive` | ¿Sobrevivió?, en texto |
| `fare` | Precio del billete (£) | `alone` | ¿Viajaba solo? |
| `embarked` | Puerto: C, Q o S | | |

> Ojo: algunas columnas esconden sorpresas.

---
## 1 · Primer contacto *(20 min)*

**Mira antes de tocar.** El objetivo es conocer el dataset, no arreglarlo.

- [ ] Primeras y últimas filas, `shape` y tipos de dato.

In [ ]:
df.shape

In [ ]:
df.head(5)

df.tail

In [ ]:
df.dtypes

- [ ] Resumen estadístico de las numéricas y de las categóricas.

In [ ]:
df.describe(exclude="number")


In [ ]:
df.describe()

- [ ] Faltantes por columna (número y %), filas duplicadas y valores únicos de cada categórica.

In [ ]:
df.nunique()

In [ ]:
faltantes = df.isna().sum()
porcentaje = (df.isna().mean() * 100).round(1)
pd.DataFrame({"faltantes": faltantes, "%": porcentaje})

In [ ]:
df.duplicated().sum()

### Preguntas de la fase 1

1. ¿Es un problema de clasificación o de regresión? ¿Por qué?
2. ¿Qué columna numérica se comporta como categórica? ¿Y qué categórica tiene orden?
3. ¿Qué parejas de columnas dicen lo mismo? **Compruébalo** con código, no a ojo.
4. ¿Qué pasa si le das `alive` al modelo?

✍️ **Tus respuestas:**

1. Es un problema de clasificacion, debido a que se hace un conteo y agrupamiento de los grupos sobrevivientes.
2. Survived y Pclass son columnas numericas que se comportan como categoricas; Class es una variable categorica que si tiene orden.
3. Como se observa en los codigos de abajo, existen 3 pares de columna identicas.
4. Habra que transformar el str a int, por lo tanto no lo digiere.

In [ ]:
df.groupby("survived")["alive"].unique()

In [ ]:
df.groupby("pclass")["class"].unique()

In [ ]:
df.groupby("embarked")["embark_town"].unique()

---
## 2 · Análisis exploratorio *(40 min)*

**Cada gráfica lleva 1–2 frases de interpretación. Sin interpretación, no cuenta.**

### Objetivo y univariado

- [ ] % de supervivientes: ¿clases equilibradas?

In [ ]:
df["survived"].value_counts(normalize=True) * 100

## ✍️Respuesta: 
Obtuvimos el conteo de la columna "Survived" para obtener el porcentaje de acuerdo a cada valor binario, viendo los porcentajes 0(Murio) = 61% y 1(Sobrevivio) = 38%, se concluye que las clases estan desequilibradas.

- [ ] Histograma y boxplot de `age` y `fare`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
sns.histplot(df["age"], ax=axes[0, 0]) 
axes[0, 0].set_title("Distribucion de Edad")
sns.boxplot(x=df["age"], ax=axes[0, 1])
axes[0, 1].set_title("BoxPlot de edad: Mediana y Valores Atipicos")
sns.histplot(df["fare"], ax=axes[1, 0])
axes[1, 0].set_title("Distribucion del precio del billete")
sns.boxplot(x=df["fare"], ax=axes[1, 1])
axes[1, 1].set_title("Boxplot de billete: Mediana y Valores Atipicos")
plt.tight_layout()
plt.show()

## Razonamiento de las graficas
En el histograma de edad, la mayoria de personas promedian 28 y son menores de 40, hay una minoria de personas en la vejez.
En el boxplot de edad, la mediana esta en 28 años y tenemos valores atipicos a partir de los 60 años.

En el histograma del precio del billete, los precios estan condensados entre 0 y 50, y la minoria de billetes cuesta mas de 100.
En el boxplot la mediana esta en aprox. 10, con muchos valores atipicos a partir de los 70.

## ✍️ Respuesta


- [ ] Barras de `sex`, `pclass`, `embarked`, `sibsp` y `parch`.

**Cómo hicimos las gráficas de barras.** Para cada columna tomamos `df["columna"]` y usamos `.value_counts()` para contar cuántos pasajeros hay de cada valor. Con `.sort_index()` ordenamos las barras por valor (0, 1, 2…) en lugar de por tamaño, y con `.plot(kind="bar")` dibujamos una barra por valor. `plt.title`, `plt.xlabel` y `plt.ylabel` ponen el título y los ejes, y `plt.show()` muestra la gráfica.

En cada gráfica, el eje horizontal tiene los valores de la columna y el vertical cuántas personas tienen ese valor.

In [ ]:
df["sex"].value_counts().sort_index().plot(kind="bar")
plt.title("Pasajeros por sex")
plt.xlabel("sex")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
df["pclass"].value_counts().sort_index().plot(kind="bar")
plt.title("Pasajeros por pclass")
plt.xlabel("pclass")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
df["embarked"].value_counts().sort_index().plot(kind="bar")
plt.title("Pasajeros por embarked")
plt.xlabel("embarked")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
df["sibsp"].value_counts().sort_index().plot(kind="bar")
plt.title("Pasajeros por sibsp(Hermanos/Conyuges)")
plt.xlabel("sibsp")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
df["parch"].value_counts().sort_index().plot(kind="bar")
plt.title("Pasajeros por parch(Padres/Hijos)")
plt.xlabel("parch")
plt.ylabel("Cantidad")
plt.show()

✍️ **Respuesta / interpretación:** (qué haces y por qué)
Se hace un recuento de cuantas personas hay en total por cada categoria. Para determinar si tienen la posibilidad de sobrevivir posteriormente

### Frente a `survived`

- [ ] Supervivencia por sexo, clase, edad, `fare` y `alone`.

**Cómo calculamos el % de supervivencia por grupo.** Como `survived` solo vale 0 o 1, su promedio es la proporción de sobrevivientes (por ejemplo, el promedio de [1, 1, 1, 0] es 0.75 = 75 %). Con `df.groupby("columna")` separamos a los pasajeros en grupos y con `["survived"].mean()` sacamos ese promedio dentro de cada grupo. Multiplicamos por 100 para tenerlo en porcentaje y usamos `display()` para mostrar las tres tablas en la misma celda.

In [ ]:
display((df.groupby("sex")["survived"].mean() * 100).round(1))
display((df.groupby("pclass")["survived"].mean() * 100).round(1))
display((df.groupby("alone")["survived"].mean() * 100).round(1))

**Gráficas de supervivencia con `sns.barplot`.** `sns.barplot(data=df, x="columna", y="survived")` calcula el promedio de `survived` para cada valor de `x`, así que muestra en gráfica los mismos porcentajes de la tabla anterior. El eje vertical va de 0 a 1 (0.7 = 70 % sobrevivió). La rayita negra sobre cada barra es el margen de incertidumbre: si las rayitas de dos barras no se tocan, la diferencia no es casualidad.

In [ ]:
sns.barplot(data=df, x="sex", y="survived")
plt.title("Supervivencia por sexo")
plt.show()

In [ ]:
sns.barplot(data=df, x="pclass", y="survived")
plt.title("Supervivencia por clase")
plt.show()

Nota Importante: Se puede observar que sobrevivio mas gente de la primera clase que de la tercera clase

In [ ]:
sns.barplot(data=df, x="alone", y="survived")
plt.title("Supervivencia si viajaba solo")
plt.show()

Nota: Podemos observar que fue mas posible la supervivencia de aquellos que viajaban acompañados

**Histograma de edad separado por supervivencia.** Con `sns.histplot(data=df, x="age")` contamos cuántas personas hay en cada rango de edad. `hue="survived"` colorea según el resultado (azul = 0, no sobrevivió; naranja = 1, sí sobrevivió) y `multiple="stack"` apila un color sobre otro, así que la altura total de cada barra son todos los pasajeros de esa edad. Para leerlo comparamos qué parte de cada barra es naranja.

In [ ]:
sns.histplot(data=df, x="age", hue="survived", multiple="stack")
plt.title("Edad según supervivencia")
plt.show()

## Analisis
Podemos observar que la mayor cantidad de fallecidos tienen en promedio 20 años, al igual que en promedio sobrevivieron personas entre 20 y 30 años.

**Boxplot de tarifa según supervivencia.** `sns.boxplot(data=df, x="survived", y="fare")` dibuja una caja para los que no sobrevivieron (0) y otra para los que sí (1). La línea dentro de la caja es la mediana, la caja contiene a la mitad central de los pasajeros y los circulitos son valores atípicos, muy alejados del resto.

In [ ]:
sns.boxplot(data=df, x="survived", y="fare")
plt.title("Tarifa según supervivencia")
plt.show()

✍️ **Respuesta / interpretación:** (qué haces y por qué)
Podemos ver que la mayor cantidad de gente murio pago menos que la gente que sobrevivio. Con muchos mas casos atipicos la gente que murio.

- [ ] ¿Influyó el puerto? **Antes de concluir, cruza puerto con clase.**

**Cómo cruzamos puerto con supervivencia y con clase.** `pd.crosstab(df["embarked"], df["survived"])` arma una tabla con los puertos en las filas y el resultado en las columnas, contando cuántas personas hay en cada casilla. Con `normalize="index"` cada fila se convierte en porcentaje (cada fila suma 100 %). La primera tabla muestra qué % de cada puerto sobrevivió y la segunda de qué clase era la gente de cada puerto (C = Cherbourg, Q = Queenstown, S = Southampton). Hicimos las dos para separar el efecto del puerto del efecto de la clase.

In [ ]:
# % de supervivencia por puerto
display((pd.crosstab(df["embarked"], df["survived"], normalize="index") * 100).round(1))

## ANALISIS
Podemos ver que la mayor cantidad de gente que sobrevivio venia del puerto C, y tanto del puerto Q como S, sobrevivieron menos. Nota: Se cree que hay casualidad y que el hecho de que sobrevivan por el puerto se debe a razones ligadas a la clase de cada tripulante

In [ ]:
# ¿Qué clase subió en cada puerto? (% de cada puerto)
display((pd.crosstab(df["embarked"], df["pclass"], normalize="index") * 100).round(1))

## Analisis
Podemos ver que la clase es la que esta ligada al puerto y es lo que determino que las personas sobrevivieran, independientemente del puerto del que salieron.

- [ ] Mujeres de 3.ª clase vs. hombres de 1.ª.

**Cómo filtramos los dos grupos.** `df["sex"] == "female"` revisa fila por fila si el pasajero es mujer y devuelve `True` o `False`. Con `&` pedimos que se cumplan las dos condiciones a la vez (cada una entre paréntesis), y `df[...]` se queda solo con esas filas. Después usamos `len()` para contar cuántas personas quedaron en cada grupo y `["survived"].mean() * 100` para sacar su % de supervivencia.

In [ ]:
mujeres_3 = df[(df["sex"] == "female") & (df["pclass"] == 3)]
hombres_1 = df[(df["sex"] == "male") & (df["pclass"] == 1)]

print("Mujeres de 3.a clase:", len(mujeres_3), "pasajeras,", round(mujeres_3["survived"].mean() * 100, 1), "% sobrevivió")
print("Hombres de 1.a clase:", len(hombres_1), "pasajeros,", round(hombres_1["survived"].mean() * 100, 1), "% sobrevivió")

✍️ **Respuesta / interpretación:**:
Se intuye que hubo un protocolo de mujeres/niños primero, ya que incluso hay en total menos mujeres que hombres, sin distinguir la clase

### Preguntas de la fase 2

- Si solo pudieras usar **una** columna para predecir, ¿cuál y por qué?
- Tres hallazgos que te sorprendan.
- ¿Algo huele a error en los datos?

✍️ **Tus respuestas:**

- Usariamos la columna Survived para predecir si sobrevivieron de acuerdo a las caracteristicas dadas por el dataset.
- Que sobrevivieron mas mujeres a pesar de haber menos, la distribucion de la gente por puerto y como coincidio con el presupuesto de la gente.
- La tasa de supervivencia de las mujeres huele a error, sospechamos de que hubo un protocolo de antemano.

---
## 3 · Limpieza *(65 min)*

- [ ] **Copia:** guarda los datos originales antes de tocar nada.

In [ ]:
df2 = df.copy()

In [ ]:
df2.head()

- [ ] **Redundancia y fuga:** elimina columnas repetidas. ¿Qué haces con `who` y `adult_male`? ¿Qué columna provoca *data leakage*?

In [ ]:
# Fuga: alive es la misma respuesta que survived escrita en texto
# Redundantes: class (= pclass), embark_town (= embarked), alone (= sibsp + parch == 0)
# who y adult_male: se deducen de sex y age
columnas_a_borrar = ["alive", "class", "embark_town", "alone", "who", "adult_male"]

df2 = df2.drop(columns=columnas_a_borrar)
print("Columnas que quedan:", list(df2.columns))
df2.shape

In [ ]:
df2.head()

✍️ **Respuesta / interpretación:** (qué haces y por qué)

De columnas repetidas tenemos. sibsp y parch con alone, class con pclass, deck y embark_town, who y adult_male con sex, alive y survived.

Creemos que teniendo el sexo y la edad de la persona es muy redundante conocer si es un hombre adulto y quien es(hombre,mujer,niño).

El data leakage o sea las respuestas se  pueden filtrar al testing, mediante las columnas survived y alive.

- [ ] **Duplicados:** no hay nombre ni billete. ¿Son errores o personas distintas?

In [ ]:
# Filas que son copia exacta de otra
df2.duplicated().sum()

In [ ]:
# Ver las filas repetidas, ordenadas para que las iguales queden juntas
df2[df2.duplicated(keep=False)].sort_values(["pclass", "sex", "fare"]).head(10)

In [ ]:
duplicados = df[df.duplicated(keep=False)]
print("Filas que forman parte de un duplicado:", len(duplicados))
duplicados.sort_values(["pclass", "sex", "fare", "age"]).head(10)

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Faltantes:** decide para `deck`, `age` y `embarked`. ¿Eliminar, media, mediana por grupo, moda o indicador?

In [ ]:
# Vacíos por columna
df2.isna().sum()

In [ ]:
# deck: 77 % vacío -> indicador 1/0 y se elimina la columna
df2["tiene_deck"] = df2["deck"].notna().astype(int)
df2 = df2.drop(columns=["deck"])

# embarked: 2 vacíos -> moda (el puerto más frecuente)
df2["embarked"] = df2["embarked"].fillna(df2["embarked"].mode()[0])

# age: 20 % vacío -> mediana de su clase y sexo
df2["age"] = df2["age"].fillna(df2.groupby(["pclass", "sex"])["age"].transform("median"))

# Comprobación: todo en 0
df2.isna().sum()

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Outliers:** revisa `fare`, `age`, `sibsp` y `parch` con IQR y Z-score. ¿Error o valor raro pero real?

In [ ]:
numericas = ["fare", "age", "sibsp", "parch"]

# IQR: atípico si está fuera de Q1 - 1.5*IQR o Q3 + 1.5*IQR
# Se usa df (original) para que las edades rellenadas no cambien el conteo
q1 = df[numericas].quantile(0.25)
q3 = df[numericas].quantile(0.75)
iqr = q3 - q1
((df[numericas] < q1 - 1.5 * iqr) | (df[numericas] > q3 + 1.5 * iqr)).sum()

In [ ]:
# Z-score: atípico si está a más de 3 desviaciones del promedio
z = (df[numericas] - df[numericas].mean()) / df[numericas].std()
(z.abs() > 3).sum()

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Antes vs. después:** compara filas, faltantes, distribuciones y % de supervivientes.

In [ ]:
print("Filas:", len(df), "->", len(df2))
print("Columnas:", df.shape[1], "->", df2.shape[1])
print("Vacíos:", df.isna().sum().sum(), "->", df2.isna().sum().sum())
print("% supervivientes:", round(df["survived"].mean() * 100, 1), "->", round(df2["survived"].mean() * 100, 1))

In [ ]:
# Distribución de la edad antes y después de rellenar
sns.histplot(df["age"], bins=30, color="gray", label="Antes")
sns.histplot(df2["age"], bins=30, alpha=0.5, label="Después")
plt.title("Edad antes y después de rellenar")
plt.legend()
plt.show()

✍️ **Respuesta / interpretación:** (qué haces y por qué)

**Pregunta clave:** billetes de 0 £ y de más de 500 £, edades menores de 1 año, 8 hermanos a bordo. ¿Los eliminarías?

In [ ]:
print("Billetes de 0 £:", (df2["fare"] == 0).sum())
print("Billetes de más de 500 £:", (df2["fare"] > 500).sum())
print("Edades menores de 1 año:", (df2["age"] < 1).sum())
print("8 hermanos o cónyuges a bordo:", (df2["sibsp"] == 8).sum())

✍️ **Tu respuesta:**

---
## 4 · Transformación y escalado *(40 min)*

- [ ] **Feature engineering:** crea ≥ 2 variables (p. ej. tamaño de familia, tramos de edad) y valida cada una contra `survived`.

In [ ]:
# Tu código aquí

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Asimetría:** aplica log a la variable que lo necesite. ¿Qué haces con los ceros?

In [ ]:
# Tu código aquí

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Codificación:** nominal → one-hot, ordinal → ordinal. ¿`pclass` como número o one-hot? ¿Cuándo usar `drop="first"`?

In [ ]:
# Tu código aquí

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Split:** 80/20 estratificado, `random_state=42`. Verifica el % de supervivientes en ambos.

In [ ]:
# Tu código aquí

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Escalado:** `StandardScaler` o `MinMaxScaler` según la forma y los outliers. `fit` **solo con train**.

In [ ]:
# Tu código aquí

✍️ **Respuesta / interpretación:** (qué haces y por qué)

- [ ] **Bonus:** todo en un `ColumnTransformer` dentro de un `Pipeline`.

In [ ]:
# Tu código aquí

**Pregunta clave:** ¿en qué orden van *split*, imputación y escalado? ¿Qué pasa si calculas la mediana de `age` con todos los datos, incluido test?

✍️ **Tu respuesta:**

### Verificación final

- [ ] Sin faltantes
- [ ] Todo numérico
- [ ] Mismas columnas en `X_train` y `X_test`
- [ ] Sin `survived` ni columnas que la revelen
- [ ] `shape` de los cuatro conjuntos (`X_train`, `X_test`, `y_train`, `y_test`)

In [ ]:
# Tu código aquí

---
## 5 · Bitácora y conclusiones *(15 min)*

### Bitácora · una fila por decisión

Mínimo: columnas eliminadas, duplicados, cada faltante, cada outlier, cada variable nueva, codificación y escalado.

| # | Columna | Problema | Opciones | Decisión | Por qué | Afectados |
|---|---------|----------|----------|----------|---------|-----------|
| 1 | *ejemplo* | 5 % faltantes | Eliminar, media, mediana | Mediana | Distribución asimétrica | 45 |
| 2 |  |  |  |  |  |  |
| 3 |  |  |  |  |  |  |
| 4 |  |  |  |  |  |  |
| 5 |  |  |  |  |  |  |

### Conclusiones

- **Los 3 factores que más influyeron:**
- **Mi decisión más difícil y qué pasaría con otra:**
- **Qué dato me faltó:**
- **Forma final y columnas que verá el modelo:**